# 01 · Collection: public procurement contracts (BASE)

**Goal:** get the public contracts for the scope set in `config.yaml` (all of Portugal, some districts, or some municipalities) into `data/raw/`, in a way that can be re-run daily.

**Source:** IMPIC publishes the complete BASE contract dataset on
[dados.gov.pt](https://dados.gov.pt/pt/datasets/contratos-publicos-portal-base-impic-contratos-de-2012-a-2026/),
one zipped JSON file per year.

**What changed from the Covilhã version**
- The municipality is no longer hard-coded: years, scope and file names come from `config.yaml` at the project root.
- With an empty scope, nothing is filtered: every contract in the national file is kept.
- The output file is `data/raw/contracts_<slug>.json` (same format as before, so `02_storage` reads it unchanged), plus an optional Parquet copy and a small `collection_manifest_<slug>.json`.

Re-running this notebook downloads a year's file again only if IMPIC has republished it.

In [ ]:
# ── Environment: Google Colab or local (VS Code / Jupyter) ─────────────────────────────
import shutil
import subprocess
import sys
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
DRIVE_FOLDER = "procurement-poc"   # folder in "My Drive" that holds the project (change for the national copy)
PIP_PACKAGES = []                  # pyyaml and pyarrow are already installed in Colab

if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    ROOT = Path("/content/drive/MyDrive") / DRIVE_FOLDER
    WORK = Path("/content/work")   # fast local disk: wiped when the session ends
    if PIP_PACKAGES:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", *PIP_PACKAGES], check=True)
else:
    ROOT = Path(".")
    WORK = ROOT / "data"

DATA = ROOT / "data"
DATA.mkdir(parents=True, exist_ok=True)
WORK.mkdir(parents=True, exist_ok=True)
print(f"running in Colab: {IN_COLAB} | project folder: {ROOT} | data folder: {DATA}")

## 0. Read the configuration
`config.yaml` must be in the project folder (next to `data/`). The cell prints the scope it will use, so you can check it before anything is downloaded.

In [ ]:
import yaml

CONFIG_FILE = ROOT / "config.yaml"
if not CONFIG_FILE.exists():
    raise FileNotFoundError(f"config.yaml not found in {ROOT}. Copy it to the project folder on Drive.")

CFG = yaml.safe_load(CONFIG_FILE.read_text(encoding="utf-8"))

SLUG = CFG["project"]["slug"]
YEARS = [int(y) for y in CFG["collection"]["years"]]
SCOPE = CFG["collection"].get("scope") or {}
COUNTRY = SCOPE.get("country")
DISTRICTS = set(SCOPE.get("districts") or [])
MUNICIPALITIES = set(SCOPE.get("municipalities") or [])
KEEP_WITHOUT_LOCATION = bool(CFG["collection"].get("keep_without_location", True))
JSON_INDENT = CFG.get("output", {}).get("json_indent")
WRITE_PARQUET = bool(CFG.get("output", {}).get("also_write_parquet", False))

NATIONAL = not DISTRICTS and not MUNICIPALITIES
print(f"slug:            {SLUG}")
print(f"years:           {YEARS}")
print(f"country:         {COUNTRY or 'any (abroad included)'}")
print(f"districts:       {sorted(DISTRICTS) or 'all'}")
print(f"municipalities:  {sorted(MUNICIPALITIES) or 'all'}")
print(f"scope:           {'ALL OF PORTUGAL (no filter)' if NATIONAL else 'filtered'}")

In [ ]:
import json
import zipfile
from datetime import datetime, timezone

import pandas as pd
import requests

DATASET_API = "https://dados.gov.pt/api/1/datasets/contratos-publicos-portal-base-impic-contratos-de-2012-a-2026/"

RAW = DATA / "raw"
RAW.mkdir(parents=True, exist_ok=True)

## 1. List the files available in the dataset
The dados.gov.pt API returns every resource with its URL and `last_modified` timestamp. The timestamp is what tells a daily run whether anything is new.

In [ ]:
meta = requests.get(DATASET_API, timeout=30).json()
zips = {r["title"]: r for r in meta["resources"] if r["format"] == "zip"}

missing = [y for y in YEARS if f"contratos{y}.zip" not in zips]
if missing:
    raise ValueError(f"no file in the dataset for year(s) {missing}")

print(meta["title"], "| update frequency:", meta["frequency"])
pd.DataFrame(
    [{"file": t, "MB": round(r["filesize"] / 1e6, 1), "last_modified": r["last_modified"][:19]} for t, r in zips.items()]
).sort_values("file", ascending=False).head(5)

## 2. Download (cached)
The file name includes IMPIC's `last_modified` stamp. If that exact file is already on disk, the download is skipped; if IMPIC has published a newer version, the older copy for that year is replaced.

In [ ]:
def download_year(year: int) -> Path:
    res = zips[f"contratos{year}.zip"]
    stamp = res["last_modified"][:19].replace("-", "").replace(":", "")
    path = RAW / f"contratos{year}_{stamp}.zip"
    if path.exists():
        print(f"{path.name}: up to date, skipping download")
        return path

    tmp = path.with_suffix(".part")
    with requests.get(res["url"], stream=True, timeout=300) as r:
        r.raise_for_status()
        with open(tmp, "wb") as f:
            for chunk in r.iter_content(1 << 20):
                f.write(chunk)
    for old in RAW.glob(f"contratos{year}_*.zip"):   # remove the old copy only after the new one is complete
        old.unlink()
    tmp.replace(path)
    print(f"{path.name}: downloaded {path.stat().st_size / 1e6:.1f} MB")
    return path


zip_paths = {year: download_year(year) for year in YEARS}

## 3. Load and apply the scope
Each zip holds one JSON array for the whole country for that year. Each execution location is a string
`"País, Distrito, Concelho"`; a contract is kept if **any** of its locations matches the scope in `config.yaml`.
With an empty scope (national), every contract is kept.

In [ ]:
def parse_location(text: str):
    parts = [p.strip() for p in (text or "").split(",", 2)]
    parts += [None] * (3 - len(parts))
    return tuple(p or None for p in parts[:3])   # (country, district, municipality)


def in_scope(locs) -> bool:
    locs = locs or []
    if not locs:
        return NATIONAL and KEEP_WITHOUT_LOCATION   # a filtered scope never keeps contracts without a place
    for text in locs:
        country, district, municipality = parse_location(text)
        if COUNTRY and country != COUNTRY:
            continue
        if NATIONAL:
            return True
        if district in DISTRICTS or municipality in MUNICIPALITIES:
            return True
    return False


def load_year(path: Path) -> pd.DataFrame:
    with zipfile.ZipFile(path) as z, z.open(z.namelist()[0]) as f:
        return pd.read_json(f, dtype={"idcontrato": "int64", "idprocedimento": "int64"})


frames, per_year = [], []
for year, path in zip_paths.items():
    df = load_year(path)
    keep = df[df["localExecucao"].map(in_scope)]
    print(f"{year}: {len(df):>7,} contracts nationally -> {len(keep):,} in scope")
    per_year.append({"year": year, "national": len(df), "in_scope": len(keep), "source_file": path.name})
    frames.append(keep)
    del df                                        # free the national frame before loading the next year

contracts = pd.concat(frames, ignore_index=True)
del frames
dups = contracts["idcontrato"].duplicated().sum()
print(f"dropping {dups} duplicated contract ids published by IMPIC")
contracts = contracts.drop_duplicates("idcontrato", keep="last").reset_index(drop=True)
print(f"memory used by the extract: {contracts.memory_usage(deep=True).sum() / 1e9:.2f} GB")
contracts.shape

In [ ]:
print(contracts["dataPublicacao"].dtype)
print(contracts["dataPublicacao"].dropna().head(5).tolist())

## 4. Quick look

In [ ]:
def parse_dates(s):
    """Dates come as dd/mm/yyyy (recent files) or yyyy-mm-dd (older files)."""
    if pd.api.types.is_datetime64_any_dtype(s):
        return s
    d = pd.to_datetime(s, format="%d/%m/%Y", errors="coerce")
    rest = d.isna() & s.notna()
    if rest.any():
        d[rest] = pd.to_datetime(s[rest], format="ISO8601", errors="coerce")
    return d


pub = parse_dates(contracts["dataPublicacao"])
print(f"{pub.isna().sum():,} contracts without a readable publication date")
print(f"published between {pub.min():%Y-%m-%d} and {pub.max():%Y-%m-%d}")
print(f"total contract value: {contracts['precoContratual'].sum():,.2f} €")
contracts[["idcontrato", "dataPublicacao", "tipoprocedimento", "objectoContrato",
           "adjudicante", "adjudicatarios", "precoContratual"]].head()

In [ ]:
pub.dt.year.value_counts().sort_index()

In [ ]:
contracts["tipoprocedimento"].value_counts().to_frame("contracts")

### Contracts by district
A contract with locations in several districts is counted once in each of them, so the column can add up to more than the total.

In [ ]:
locs = (
    contracts[["idcontrato", "localExecucao", "precoContratual"]]
    .explode("localExecucao")
    .assign(district=lambda d: d["localExecucao"].map(lambda t: parse_location(t)[1] if isinstance(t, str) else None))
    .drop_duplicates(["idcontrato", "district"])
)
by_district = (
    locs.groupby(locs["district"].fillna("(sem distrito)"))
    .agg(contracts=("idcontrato", "nunique"), value_eur=("precoContratual", "sum"))
    .sort_values("contracts", ascending=False)
)
by_district["value_eur"] = by_district["value_eur"].round(0)
print(f"{contracts['localExecucao'].map(lambda l: not l).sum():,} contracts have no execution location")
by_district

### Sanity check: Covilhã still matches the old extract

1.   List item
2.   List item


If the old file `contracts_covilha.json` is still in `data/raw/`, the Covilhã subset of the new extract should contain the same contracts (or more, if IMPIC published new ones since).

old_file = RAW / "contracts_covilha.json"
if old_file.exists() and SLUG != "covilha":
    old_ids = {r["idcontrato"] for r in json.loads(old_file.read_text(encoding="utf-8"))}
    cov = contracts["localExecucao"].map(lambda l: any(parse_location(t)[2] == "Covilhã" for t in (l or [])))
    new_ids = set(contracts.loc[cov, "idcontrato"])
    print(f"Covilhã: old extract {len(old_ids):,} | new extract {len(new_ids):,}")
    print(f"  in old but missing now: {len(old_ids - new_ids):,}   (should be 0, or a few republished/removed by IMPIC)")
    print(f"  new since the old extract: {len(new_ids - old_ids):,}")
else:
    print("no old Covilhã extract to compare with — skipped")

## 5. Save the raw extract
Records are kept exactly as IMPIC publishes them, with list fields left as lists and nothing renamed. All cleaning happens in `02_storage.ipynb`.
The file is written to a temporary name first and then renamed, so an interrupted run never leaves a half-written file for 02.

In [ ]:
out = RAW / f"contracts_{SLUG}.json"
tmp = out.with_suffix(".json.part")
contracts.to_json(tmp, orient="records", force_ascii=False, indent=JSON_INDENT)
tmp.replace(out)
print(f"saved {len(contracts):,} contracts -> {out}  ({out.stat().st_size / 1e6:.1f} MB)")

if WRITE_PARQUET:
    pq = RAW / f"contracts_{SLUG}.parquet"
    try:
        contracts.to_parquet(pq, index=False)
        print(f"saved Parquet copy -> {pq}  ({pq.stat().st_size / 1e6:.1f} MB)")
    except Exception as e:                       # mixed-type columns can break Parquet; the JSON is what 02 uses
        print(f"Parquet copy skipped: {e}")

manifest = {
    "slug": SLUG,
    "run_at": datetime.now(timezone.utc).isoformat(timespec="seconds"),
    "scope": {"national": NATIONAL, "country": COUNTRY,
              "districts": sorted(DISTRICTS), "municipalities": sorted(MUNICIPALITIES)},
    "years": per_year,
    "contracts": int(len(contracts)),
    "published_from": f"{pub.min():%Y-%m-%d}",
    "published_to": f"{pub.max():%Y-%m-%d}",
    "total_value_eur": float(contracts["precoContratual"].sum()),
}
(RAW / f"collection_manifest_{SLUG}.json").write_text(json.dumps(manifest, ensure_ascii=False, indent=1), encoding="utf-8")
manifest

### Note for later (03: documents)
`linkPecasProc` links to the procedure documents. With the national scope, **03 must not try to fetch all of them**: it will follow `documents.scope` in `config.yaml` (Step 4 of the plan).

has_docs = contracts["linkPecasProc"].fillna("").str.startswith("http")
print(f"{has_docs.sum():,} of {len(contracts):,} contracts have a procedure-documents link")
contracts.loc[has_docs, "linkPecasProc"].str.extract(r"https?://([^/]+)")[0].value_counts().head(10).to_frame("contracts by platform")